# EmotiGuard: Emotion-Refusal Vector Interaction in LLM Safety

## Research Question
Do emotion-like activation patterns interact with and modulate refusal mechanisms in large language models?

This notebook implements the full experimental pipeline:
1. Extract refusal vectors (Arditi et al., 2024)
2. Extract emotion vectors (inspired by Anthropic, April 2026)
3. Analyze geometric interaction
4. Run causal steering experiments
5. Evaluate safety implications

In [ ]:
import sys
sys.path.insert(0, '../..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from src.model_adapter import ModelAdapter, ModelConfig
from src.vector_extraction import ExtractionConfig, VectorExtractor, compute_interaction_metrics, find_best_layer
from src.steering import SteeringConfig, SteeringExperiment
from src.evaluation import compute_safety_report, compare_conditions
from src.prompts import (
    HARMFUL_PROMPTS, HARMLESS_PROMPTS, EMOTION_PROMPTS,
    NEUTRAL_PROMPTS, TEST_HARMFUL_PROMPTS,
)
from src.visualization import (
    plot_cosine_similarity_heatmap, plot_refusal_sweep,
    plot_multi_emotion_sweep, plot_combined_steering,
    plot_stealth_analysis, plot_vector_geometry_2d
)

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Phase 1: Vector Extraction

### 1a. Setup and Model Loading

In [ ]:
# Load model via ModelAdapter
adapter = ModelAdapter(ModelConfig(
    name='meta-llama/Llama-3.1-8B-Instruct',
    device='cuda',
    dtype=torch.float16,
    batch_size=4,
))

# Configure extraction
config = ExtractionConfig(
    target_layers=list(range(adapter.num_layers)),
    batch_size=4,
)
extractor = VectorExtractor(adapter, config)

### 1b. Extract Refusal Vectors

In [ ]:
# Format prompts using adapter's chat template
harmful_formatted = adapter.format_prompts(HARMFUL_PROMPTS)
harmless_formatted = adapter.format_prompts(HARMLESS_PROMPTS)

# Extract refusal direction at each layer
refusal_vectors = extractor.extract_refusal_vector(
    harmful_formatted, harmless_formatted
)

print(f'Extracted refusal vectors for {len(refusal_vectors)} layers')
print(f'Vector dimensionality: {refusal_vectors[0].shape}')

### 1c. Extract Emotion Vectors

In [ ]:
# Option A: Story-based extraction (Anthropic's method — PRIMARY)
# Requires stories to be generated first: python cli.py generate-stories
USE_STORIES = False  # Set to True after generating stories

if USE_STORIES:
    from src.story_generator import load_stories
    stories = load_stories('../../data/stories')
    emotion_vectors = extractor.extract_emotion_vectors(
        stories,
        use_cross_emotion_baseline=True,
        token_mode='mean_from_n',
        mean_from_token=50,
    )
else:
    # Option B: Descriptive extraction (simple baseline for comparison)
    emotion_formatted = {
        emotion: adapter.format_prompts(prompts)
        for emotion, prompts in EMOTION_PROMPTS.items()
    }
    neutral_formatted = adapter.format_prompts(NEUTRAL_PROMPTS)
    emotion_vectors = extractor.extract_emotion_vectors(
        emotion_formatted, neutral_prompts=neutral_formatted
    )

print(f'Extracted vectors for {len(emotion_vectors)} emotions')
for emotion in emotion_vectors:
    print(f'  - {emotion}: {len(emotion_vectors[emotion])} layers')

### 1d. Save Vectors for Later Use

In [ ]:
import os
os.makedirs('../results', exist_ok=True)

torch.save(refusal_vectors, '../results/refusal_vectors.pt')
torch.save(emotion_vectors, '../results/emotion_vectors.pt')
print('Vectors saved to ../results/')

## Phase 2: Geometric Analysis

### Key Question: How do emotion vectors relate to the refusal direction?

If an emotion vector has **negative cosine similarity** with the refusal vector,
it points in the opposite direction — potentially *suppressing* refusal.
This would be the geometric signature of emotion-based safety erosion.

In [ ]:
# Compute interaction metrics
metrics = compute_interaction_metrics(refusal_vectors, emotion_vectors)

# Heatmap of cosine similarities
fig = plot_cosine_similarity_heatmap(
    metrics,
    save_path='../results/figures/emotion_refusal_heatmap.png'
)
plt.show()

In [ ]:
# Find the most interesting layers for each emotion
print('Peak interaction layers (highest |cosine similarity| with refusal):')
print('=' * 60)
for emotion in emotion_vectors:
    best_layer = find_best_layer(refusal_vectors, emotion_vectors, emotion)
    cos_sim = metrics[emotion][best_layer]['cosine_similarity']
    print(f'{emotion:>15}: layer {best_layer:>2} (cos_sim = {cos_sim:+.4f})')

In [ ]:
# 2D PCA visualization of vector geometry
# Pick the layer with strongest desperation-refusal interaction
analysis_layer = find_best_layer(refusal_vectors, emotion_vectors, 'desperation')
print(f'Using layer {analysis_layer} for geometric visualization')

ref_vec_np = refusal_vectors[analysis_layer].numpy()
emo_vecs_np = {
    e: emotion_vectors[e][analysis_layer].numpy()
    for e in emotion_vectors
}

fig = plot_vector_geometry_2d(
    ref_vec_np, emo_vecs_np,
    save_path='../results/figures/vector_geometry_pca.png'
)
plt.show()

## Phase 3: Causal Steering Experiments

### The Critical Test
Does steering with emotion vectors actually change refusal behavior?
Geometric alignment is suggestive, but causation requires intervention.

In [ ]:
# Set up the steering experiment
experiment = SteeringExperiment(
    adapter=adapter,
    refusal_vectors=refusal_vectors,
    emotion_vectors=emotion_vectors,
)

test_prompts = adapter.format_prompts(TEST_HARMFUL_PROMPTS)
print(f'Test prompts: {len(test_prompts)}')

In [ ]:
# Run desperation sweep — the headline experiment
desp_layer = find_best_layer(refusal_vectors, emotion_vectors, 'desperation')
print(f'Running desperation sweep at layer {desp_layer}')

desperation_sweep = experiment.run_emotion_refusal_sweep(
    harmful_prompts=test_prompts,
    emotion='desperation',
    layer=desp_layer,
    strengths=[-3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 3.0, 5.0],
)

fig = plot_refusal_sweep(
    desperation_sweep,
    save_path='../results/figures/desperation_sweep.png'
)
plt.show()

In [ ]:
# Run sweeps for all emotions
all_sweeps = {}
for emotion in ['desperation', 'calm', 'anger', 'fear', 'guilt', 'confidence']:
    layer = find_best_layer(refusal_vectors, emotion_vectors, emotion)
    print(f'\nSweeping {emotion} at layer {layer}...')
    all_sweeps[emotion] = experiment.run_emotion_refusal_sweep(
        harmful_prompts=test_prompts,
        emotion=emotion,
        layer=layer,
        strengths=[-2.0, -1.0, 0.0, 1.0, 2.0, 3.0],
    )

fig = plot_multi_emotion_sweep(
    all_sweeps,
    save_path='../results/figures/all_emotions_sweep.png'
)
plt.show()

## Phase 4: Stealth Compliance Analysis

### The Most Dangerous Finding
From Anthropic's paper: desperation-steered compliance can occur
**without visible emotional markers** in the output. The model
looks calm and competent while its integrity is compromised.

Can we replicate this? And can we detect it?

In [ ]:
# Compare baseline vs steered responses
from src.evaluation import SafetyReport

reports = []

# Baseline
baseline = experiment.measure_refusal_rate(test_prompts, configs=[])
reports.append(compute_safety_report(baseline['responses'], 'Baseline'))

# Desperation steered
desp_configs = [SteeringConfig(
    vector=emotion_vectors['desperation'][desp_layer],
    layer=desp_layer, strength=3.0, method='add'
)]
desp_result = experiment.measure_refusal_rate(test_prompts, configs=desp_configs)
reports.append(compute_safety_report(desp_result['responses'], 'Desperation (α=3)'))

# Calm reduced
calm_layer = find_best_layer(refusal_vectors, emotion_vectors, 'calm')
calm_configs = [SteeringConfig(
    vector=emotion_vectors['calm'][calm_layer],
    layer=calm_layer, strength=-3.0, method='add'
)]
calm_result = experiment.measure_refusal_rate(test_prompts, configs=calm_configs)
reports.append(compute_safety_report(calm_result['responses'], 'Reduced Calm (α=-3)'))

# Print comparison
print(compare_conditions(reports))

In [ ]:
# Visualize stealth compliance breakdown
fig = plot_stealth_analysis(
    reports,
    save_path='../results/figures/stealth_analysis.png',
)
plt.show()

## Phase 5: Defense — Can We Counter Emotion-Based Attacks?

If emotion steering erodes safety, can we defend against it by
amplifying the refusal vector? This tests whether safety can
be *hardened* against emotional pressure.

In [ ]:
# Combined steering: desperation + refusal amplification
combined_results = experiment.run_combined_steering(
    harmful_prompts=test_prompts,
    emotion='desperation',
    layer=desp_layer,
    emotion_strength=3.0,
    refusal_strengths=[0.0, 1.0, 2.0, 3.0, 5.0, 8.0],
)

fig = plot_combined_steering(
    combined_results,
    save_path='../results/figures/defense_combined.png',
)
plt.show()

## Summary & Key Findings

Fill in after running experiments:

1. **Geometric relationship**: [Do emotion vectors align/oppose the refusal direction?]
2. **Causal effect**: [Does emotion steering change refusal rates?]
3. **Stealth compliance**: [Can we replicate invisible safety erosion?]
4. **Defense**: [Can refusal amplification counteract emotion-based attacks?]

### Implications for AI Safety
- Competence and reliability metrics would NOT catch emotion-based safety erosion
- Internal state monitoring (traceability) is essential for detecting stealth compliance
- Integrity and benevolence cannot be reduced to behavioral rules — they involve internal representations

### Connection to Trustworthiness Framework
This work demonstrates why the four-criteria trustworthiness model
(competence, reliability, integrity, benevolence) requires ALL four dimensions.
A model can be competent and reliable while its integrity is being eroded
by internal emotional dynamics — and without traceability, we'd never know.